# multilingual-e5-large + UMAP + HDBSCAN

`multilingual-e5-large` is the finalized embedding model for this project. This notebook pairs it
with **HDBSCAN** — the clustering algorithm used everywhere else in this project so far — as the
baseline in the algorithm comparison. Same settings (`min_cluster_size=2, min_samples=1,
cluster_selection_method="eom"`) used throughout.

Note: unlike `notebooks/clustering_e5.ipynb` (which clustered the raw 1024-d embeddings directly),
this notebook reduces to 5-D with UMAP first — matching the pipeline in every other notebook in
`Aggregation_Pipeline/clustering/` so the algorithm comparison is apples-to-apples. That means this
notebook's result is not directly comparable to `clustering_e5.ipynb`'s original HDBSCAN result;
the difference is UMAP, not HDBSCAN.

In [1]:
!pip install -q hdbscan umap-learn

In [2]:
# Paths (Kaggle)
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("/kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data")
RESULTS_DIR = Path("/kaggle/working/results/e5_hdbscan")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Results dir: {RESULTS_DIR.resolve()}")

Data dir: /kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data
Results dir: /kaggle/working/results/e5_hdbscan


In [3]:
# Load annotator files
df_a = pd.read_csv(DATA_DIR / "a.csv")
df_d = pd.read_csv(DATA_DIR / "d.csv")
df_p = pd.read_csv(DATA_DIR / "p.csv")

for frame in (df_a, df_d, df_p):
    frame.drop(columns=["bias_label"], inplace=True, errors="ignore")

print(df_a.shape, df_d.shape, df_p.shape)

(750, 6) (750, 8) (800, 6)


In [4]:
# Concatenate dataframes and drop duplicate articles by article_id
df = pd.concat([df_a, df_d, df_p], ignore_index=True, sort=False)
df = df.drop_duplicates(subset="article_id", keep="first").reset_index(drop=True)
df.drop(columns=["flags", "Unnamed: 8"], inplace=True, errors="ignore")

print("df shape:", df.shape)
df.head()

df shape: (2000, 6)


,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [5]:
# Keep rows with the fields required for clustering
required_columns = ["article_id", "publisher", "url", "published_at", "title", "body_text"]
df = df.dropna(subset=required_columns).copy()

print("df shape after required-field dropna:", df.shape)
df.info()

df shape after required-field dropna: (1999, 6)
<class 'pandas.core.frame.DataFrame'>
Index: 1999 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   article_id    1999 non-null   object
 1   publisher     1999 non-null   object
 2   url           1999 non-null   object
 3   published_at  1999 non-null   object
 4   title         1999 non-null   object
 5   body_text     1999 non-null   object
dtypes: object(6)
memory usage: 109.3+ KB


In [6]:
# Unicode normalization (NFC - canonical decomposition + composition)
for column in ["title", "body_text"]:
    df[column] = df[column].astype(str).map(lambda value: unicodedata.normalize("NFC", value))

df.head()

,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [7]:
# Remove title duplication from the beginning of the body text
def remove_title_from_body(row):
    body = row["body_text"].strip()
    title = row["title"].strip()
    if body.startswith(title):
        body = body[len(title):].lstrip("\n").lstrip()
    return body


df["text"] = df.apply(remove_title_from_body, axis=1)
df[["title", "text"]].head()

,title,text
0,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහ...
1,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',"මෙහි කිසිවක් අඩංගු නැත.Play video, ""දිරිය මිනි..."
2,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,සිගිරිය පිදුරංගල මාර්ගයේදී ඊයේ සවස වන අලියෙකු ...
3,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් දෙ...
4,ස්ථාන දෙකකදී ඝාතන දෙකක්,කුලී නිවසක පදිංචිව සිටි පුද්ගලයෙකුව ඊයේ (18) ර...


In [8]:
# Whitespace normalization
df["text"] = df["text"].map(lambda value: re.sub(r"\n{2,}", "\n", value))
df["text"] = df["text"].map(lambda value: re.sub(r"[ \t]+", " ", value))
df["text"] = df["text"].str.strip()

df = df[df["text"].str.len() > 0].reset_index(drop=True)
print("df shape after text cleaning:", df.shape)
df.head()

df shape after text cleaning: (1999, 7)


,article_id,publisher,url,published_at,title,body_text,text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...,දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...,"මෙහි කිසිවක් අඩංගු නැත.Play video, ""දිරිය මිනි..."
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...,සිගිරිය පිදුරංගල මාර්ගයේදී ඊයේ සවස වන අලියෙකු ...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් දෙ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...,කුලී නිවසක පදිංචිව සිටි පුද්ගලයෙකුව ඊයේ (18) ර...


In [9]:
# Build documents for embedding (title + body), with the "passage: " prefix multilingual-e5
# models require for corpus-side text
def build_passage_text(title, body):
    title = str(title).strip() if title else ""
    body = str(body).strip() if body else ""
    combined = f"{title}. {body}" if title else body
    return "passage: " + combined


df["passage_text"] = df.apply(lambda row: build_passage_text(row["title"], row["text"]), axis=1)
print(f"Documents: {len(df)}")
df["passage_text"].iloc[0][:500]

Documents: 1999


'passage: දැන් තෝරු-මෝරු අහුවෙන කාලේ. දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහනුවර දිස්ත්\u200dරික් මන්ත්\u200dරී ජගත් මනුවර්ණ මහතා:-(ජා.ජ.බ) පාර්ලිමේන්තුවේදී පැවසීය.\nදූෂණයට විරුද්ධ වීම භයානක බවත් දූෂණයට විරුද්ධ නොවී සිටීම ඊට වඩා භයානක බවත් අනුර දිසානායක ජනාධිපති\xa0 එක්සත් ජාතීන්ගේ මහා මණ්ඩලයේ අමතමින් ප්\u200dරකාශ කළා.එය අප නැවත අවධාරණය කළ යුතුයි.අපි දේශපාලන පලි ගැනීම් කරනවා යැයි චෝදනා කරනවා.නමුත් ඇත්ත ඒක නෙමෙයි.මත්තල ගුවන් තොටුපොලේ මගින් පර්යන්තය එක\xa0 ආණ්ඩුවක් නෙළුම් පොහොට්ටුවක හැඩයකට හදන්න තීරණය කළාම ඊට පස'

In [10]:
# Load multilingual-e5-large (the finalized embedding model for this project)
import torch
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model_name = "intfloat/multilingual-e5-large"
tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
model = AutoModel.from_pretrained(embedding_model_name).to(device)
model.eval()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 1024, padding_idx=1)
    (token_type_embeddings): Embedding(1, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 1024, padding_idx=1)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-23): 24 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwi

In [11]:
# Mean pooling — established as the best strategy for this model on this corpus in
# notebooks/clustering_e5.ipynb (separation_score 0.2130 for mean vs 0.1264 for max pooling)
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def embed_passages(texts, batch_size=16, max_length=512):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt",
        ).to(device)
        output = model(**encoded)
        pooled = mean_pooling(output.last_hidden_state, encoded["attention_mask"])
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0).numpy()


embeddings = embed_passages(df["passage_text"].tolist(), batch_size=16)
print(embeddings.shape)

(1999, 1024)


In [12]:
# Embedding separation score (comparable across notebooks): 1 - mean pairwise cosine similarity
# on a random 100-document sample of the raw embeddings.
from sklearn.metrics.pairwise import cosine_similarity

rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(embeddings), size=min(100, len(embeddings)), replace=False)
sample_emb = embeddings[sample_idx]

sims = cosine_similarity(sample_emb)
pairwise = sims[np.triu_indices_from(sims, k=1)]
separation_score = 1 - pairwise.mean()

print(f"Embedding model: {embedding_model_name}")
print(f"mean_sim={pairwise.mean():.4f} std={pairwise.std():.4f} min={pairwise.min():.4f} max={pairwise.max():.4f}")
print(f"Separation score (1 - mean cosine similarity): {separation_score:.4f}")

Embedding model: intfloat/multilingual-e5-large
mean_sim=0.7909 std=0.0243 min=0.7123 max=0.9162
Separation score (1 - mean cosine similarity): 0.2091


In [13]:
# Reduce dimensionality with UMAP (same settings used across this project's UMAP-based
# notebooks, e.g. notebooks/clustering_BGE_M3_BERTopic.ipynb)
from umap import UMAP

umap_model = UMAP(
    n_neighbors=3,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED,
)
reduced_embeddings = umap_model.fit_transform(embeddings)
print(reduced_embeddings.shape)

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


(1999, 5)


In [14]:
# HDBSCAN clustering
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=2,
    min_samples=1,
    metric="euclidean",
    cluster_selection_method="eom",
)
df["cluster_id"] = clusterer.fit_predict(reduced_embeddings)

print(df["cluster_id"].value_counts().head(20))
print("Noise (unclustered) articles:", int((df["cluster_id"] == -1).sum()))
print("Number of clusters found:", df["cluster_id"].nunique() - (1 if -1 in df["cluster_id"].values else 0))

cluster_id
-1      159
 168     10
 560     10
 238      9
 529      9
 210      9
 164      8
 361      8
 39       8
 326      7
 234      7
 282      7
 305      7
 431      7
 188      6
 198      6
 524      6
 364      6
 115      6
 504      6
Name: count, dtype: int64
Noise (unclustered) articles: 159
Number of clusters found: 587


In [15]:
extra_row_fields = {}

In [16]:
# Clustering evaluation
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

labels = df["cluster_id"].values
mask = labels != -1  # -1 = noise; only meaningful for density-based algorithms
X_valid = reduced_embeddings[mask]
labels_valid = labels[mask]
n_clusters = len(set(labels_valid))
noise_ratio = 1 - mask.mean()

print(f"Model: {embedding_model_name} + UMAP + HDBSCAN")
print(f"Articles: {len(df)} | Clusters (excl. noise): {n_clusters} | Noise ratio: {noise_ratio:.2%}")

if n_clusters > 1:
    sil = silhouette_score(X_valid, labels_valid, metric="cosine")
    dbi = davies_bouldin_score(X_valid, labels_valid)
    ch = calinski_harabasz_score(X_valid, labels_valid)
    print(f"Silhouette Score (cosine): {sil:.4f}")
    print(f"Davies-Bouldin Index:      {dbi:.4f}  (lower is better)")
    print(f"Calinski-Harabasz Index:   {ch:.2f}  (higher is better)")
else:
    sil = dbi = ch = float("nan")
    print("Not enough clusters to compute silhouette/DBI/CH.")

scores_path = RESULTS_DIR / "e5_hdbscan_scores.csv"
row = {
    "model": embedding_model_name,
    "pipeline": "UMAP+HDBSCAN",
    "embedding_dim": embeddings.shape[1],
    "separation_score": round(separation_score, 4),
    "n_articles": len(df),
    "n_clusters": n_clusters,
    "noise_ratio": round(noise_ratio, 4),
    "silhouette": round(sil, 4) if n_clusters > 1 else None,
    "davies_bouldin": round(dbi, 4) if n_clusters > 1 else None,
    "calinski_harabasz": round(ch, 2) if n_clusters > 1 else None,
}
row.update(extra_row_fields)
pd.DataFrame([row]).to_csv(scores_path, index=False)
print(f"Saved scores to {scores_path}")

Model: intfloat/multilingual-e5-large + UMAP + HDBSCAN
Articles: 1999 | Clusters (excl. noise): 587 | Noise ratio: 7.95%
Silhouette Score (cosine): 0.8197
Davies-Bouldin Index:      0.3501  (lower is better)
Calinski-Harabasz Index:   23379.55  (higher is better)
Saved scores to /kaggle/working/results/e5_hdbscan/e5_hdbscan_scores.csv


In [17]:
# Inspect sample titles per cluster
for cluster_id, group in list(df[df["cluster_id"] != -1].groupby("cluster_id"))[:10]:
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(10):
        print(f"- {title}")
    print()

=== Cluster 0 (4 articles) ===
- කෙන්යාවේ ඛේදවාචකයක්
- ඇෆ්ගනිස්තානයට යළිත් ගංවතුර - ජීවිත රැසක් අහිමි වෙයි
- උතුරු සුඩානයේ ගැටුම් තීව්‍ර වෙයි
- දකුණු කොරියාවේ ගංවතුර සහ නායයෑම් හේතුවෙන් පුද්ගලයින් 14 දෙනෙකු මරුට

=== Cluster 1 (3 articles) ===
- පේරේ මුහුණ දුන් තත්ත්වය උප කුලපති කියයි
- ජල ගැල්මෙන් පේරාදෙණිය  සරසවියට කෝටි 300ක පාඩුවක්
- ජලයට කොටූ  පේරේ සිසුන් දුෂ්කරතාවන්ට

=== Cluster 2 (3 articles) ===
- උත්සව සමයේ බිත්තර මිල අඩුවෙයිද?
- බිත්තරයක් රුපියල් 10කින් පහළට
- බේකරි නිෂ්පාදන මිල ගණන් ගැන තීරණයක්

=== Cluster 3 (3 articles) ===
- කැනඩා රැකියා සිහින පෙන්වූ ජාවාරම්කරුවෝ
- රුමේනියා රැකියා පෙන්වා මුදල් වංචා කළ සහකාර කළමනාකරුවකු අල්ලයි
- රට රස්සා දෙනවා කියා කෝටි 74 කට පොලු

=== Cluster 4 (3 articles) ===
- සුබ නැකතට රට අනුරට දුන් අය විපතකට
- ආණ්ඩුව මැවූ දිව්‍යලෝකය කඩා වැටිලා
- මාලිමාවත් ගොවියා මංමුලා කළා

=== Cluster 5 (3 articles) ===
- Fox Hill ධාවන ශූරයෙක් වූ බුද්ධික ගැන සංවේදී කතාවක්
- දියතලාව බිහිසුණු ඛේදවාචකයේ නොදත් කතාව
- Fox Hill අනතුර: සාමාන්‍ය මාර්ග නීති මෝටර් රථ ධාවන තරග

In [18]:
# Inspect a RANDOM sample of clusters (rather than just the first few by id) — a more
# representative check of overall cluster quality than always looking at the same low-numbered
# clusters
rng_inspect = np.random.default_rng(SEED)
cluster_ids = df.loc[df["cluster_id"] != -1, "cluster_id"].unique()
sample_size = min(8, len(cluster_ids))
sampled_cluster_ids = rng_inspect.choice(cluster_ids, size=sample_size, replace=False)

for cluster_id in sampled_cluster_ids:
    group = df[df["cluster_id"] == cluster_id]
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(15):
        print(f"- {title}")
    print()

=== Cluster 451 (5 articles) ===
- ගත වූ මාස කිහිපය තුළ රිය අනතුරුවලින් ජීවිත 1,417ක් මරුට
- බීලා සුක්කානම ගත්තොත් වැඩ වරදී
- ඉහළ යන රිය අනතුරු නිසා රටේ ආර්ථිකයට ඇතිවන බලපෑම ගැන ඔබ දන්නවා ද?
- පොලිස් දෙපාර්තමේන්තුවට රියැදුරන් බීමත් දැයි පරීක්ෂා කෙරෙන බැලුන් වල හිඟයක් තිබේ ද? : මහජන ආරක්ෂක ඇමති බීබීසීයට පැවසූ කරුණු
- අවම වේග සීමාවන්ද ඇතුළත්ව අධිවේගී මාර්ගවලට නව නීති මාලාවක්

=== Cluster 376 (3 articles) ===
- ඔබේ බැංකු තැන්පතුවලින් අය කෙරෙන රැඳවුම් බද්දට ආණ්ඩුව කරන්න යන්නේ කුමක් ද?
- Paye Tax නව වෙනස ඔබේ වැටුපට බලපාන්නේ කොහොම ද?
- රනිල්, සජිත්, අනුර, නාමල් යටතේ Paye Tax එකට මොකද වෙන්නේ ?

=== Cluster 121 (2 articles) ===
- ටින් මාළු සඳහා පාලන මිලක්
- මස් හා මාළු මිල ඉහළට

=== Cluster 412 (3 articles) ===
- රට ගොඩ නගන්න ඉන්ද්‍රා ට්‍රේඩර්ස් කෝටි දහයක් දෙයි
- රට නගන අරමුදලට තවත් ඩොලර් 16,000ක්
- රට නගන්න ප්‍රාදේශීය සංවර්ධන බැංකුවෙන් මිලියන 2.5ක්

=== Cluster 160 (5 articles) ===
- එක්සත් රාජධානියේ කැරලිවලට මැදි වූ ලාංකිකයෙකුගේ අත්දැකීම
- පවුල් ජීවිත අවුල් කරන අසීමාන්තික මව-පුතු  බැඳීම
- සා

In [19]:
# Save article-level assignments
assignments_path = RESULTS_DIR / "e5_hdbscan_assignments.csv"
df.drop(columns=["passage_text"], errors="ignore").to_csv(assignments_path, index=False, encoding="utf-8-sig")
print(f"Saved assignments: {assignments_path.resolve()}")

Saved assignments: /kaggle/working/results/e5_hdbscan/e5_hdbscan_assignments.csv


In [ ]:
# Print every cluster (capped at 20 titles each, so this stays readable even when there are
# hundreds of clusters)
for cluster_id, group in sorted(
    df[df["cluster_id"] != -1].groupby("cluster_id"), key=lambda item: item[0]
):
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(20):
        print(f"- {title}")
    if len(group) > 20:
        print(f"... ({len(group) - 20} more)")
    print()


: 